## FULL PIPELINE CODE

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

## LOADING DATASET

In [3]:
import os
os.getcwd()

'/Users/abhisheksmac'

In [4]:
os.chdir('/Users/abhisheksmac/Desktop/IT Career Switch/Portfolio Project Income Classification/')

In [9]:
df = pd.read_csv("income_evaluation.csv")  # adjust name if different
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [12]:
df.columns = df.columns.str.strip()

In [13]:
df = df.replace("?", np.nan)

## Split Features & Target

In [14]:
X = df.drop("income", axis=1)
y = df["income"]

## Train-Test Split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# PREPROCESSING PIPELINE

## Identify column types

In [16]:
categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

## Numeric Pipeline

In [17]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## Categorical Pipeline

In [18]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

## Combine Preprocessing

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# MODEL PIPELINE

In [20]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# TRAIN MODEL

In [21]:
clf.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# PREDICTIONS

In [22]:
y_pred = clf.predict(X_test)

# EVALUATION

In [23]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.857362198679564
              precision    recall  f1-score   support

       <=50K       0.89      0.93      0.91      4945
        >50K       0.73      0.64      0.68      1568

    accuracy                           0.86      6513
   macro avg       0.81      0.78      0.80      6513
weighted avg       0.85      0.86      0.85      6513



# FEATURE IMPORTANCE

In [24]:
# Get feature names
ohe = clf.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]

cat_features = ohe.get_feature_names_out(categorical_cols)
all_features = np.concatenate([numerical_cols, cat_features])

importances = clf.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": all_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

feature_importance_df.head(10)

,Feature,Importance
1,fnlwgt,0.160089
0,age,0.151507
3,capital-gain,0.091732
5,hours-per-week,0.081904
2,education-num,0.066321
33,marital-status_ Married-civ-spouse,0.062905
53,relationship_ Husband,0.042542
4,capital-loss,0.028857
35,marital-status_ Never-married,0.022578
42,occupation_ Exec-managerial,0.018738
